In [2]:
%pip install sentence-transformers torch

   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/596.4 kB ? eta -:--:--
   ---------------------------------------- 596.4/596.4 kB 2.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.6 MB 2.5 MB/s eta 0:00:05
   ---- ----------------------------------- 1.3/11.6 MB 3.4 MB/s eta 0:00:04
   ------ --------------------------------- 1.8/11.6 MB 3.0 MB/s eta 0:00:04
   -------- ------------------------------- 2.4/11.6 MB 2.9 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/11.6 MB 3.0 MB/s eta 0:00:03
   ------------- -------------------------- 3.9/11.6 MB 3.2 MB/s eta 0:00:03
   ---------------- ----------------------- 4.7/11.6 MB 3.3 MB/s eta 0:00:03
   ------------------ --------------------- 5.5/11.6 MB 3.4 MB/s eta 0:00:02
   --------------------- ------------------ 6.3/11.6 MB 3.4 MB/s eta 0:00:02
   ----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cli-base 0.6.0 requires click<8.2, but you have click 8.4.2 which is incompatible.


In [4]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch

# Load the lightweight, high-performance Transformer model
print("Loading all-MiniLM-L6-v2 model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")

Loading all-MiniLM-L6-v2 model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\omp72\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [7]:
q1 = "How can I lose weight fast?"
q2 = "What is the best way to reduce body fat?"

# Generate 384-dimensional embeddings
emb1 = model.encode(q1, convert_to_tensor=True)
emb2 = model.encode(q2, convert_to_tensor=True)

# Calculate Cosine Similarity using PyTorch / SentenceTransformer util
similarity = util.cos_sim(emb1, emb2).item()

print("Question 1:", q1)
print("Question 2:", q2)
print(f"\nSentenceTransformer Similarity Score: {similarity:.4f} ({similarity*100:.1f}%)")

Question 1: How can I lose weight fast?
Question 2: What is the best way to reduce body fat?

SentenceTransformer Similarity Score: 0.6032 (60.3%)


In [8]:
q1_jap = "When do you use シ instead of し?"
q2_eng = "When do you use \"&\" instead of \"and\"?"

emb_jap = model.encode(q1_jap, convert_to_tensor=True)
emb_eng = model.encode(q2_eng, convert_to_tensor=True)

similarity_jap = util.cos_sim(emb_jap, emb_eng).item()

print("Question 1:", q1_jap)
print("Question 2:", q2_eng)
print(f"\nSentenceTransformer Similarity Score: {similarity_jap:.4f} ({similarity_jap*100:.1f}%)")

Question 1: When do you use シ instead of し?
Question 2: When do you use "&" instead of "and"?

SentenceTransformer Similarity Score: 0.3227 (32.3%)


In [9]:
# Load dataset
df = pd.read_csv('C:\\Users\\omp72\\Desktop\\Semantic Duplicate Ques Analysis Project\\data\\quora_ques.csv').dropna(subset=['question1', 'question2'])

# Extract unique questions list from dataset (take 5,000 for quick testing)
questions_list = list(set(df['question1'].head(2500).tolist() + df['question2'].head(2500).tolist()))
print(f"Total Unique Database Questions: {len(questions_list)}")

print("Generating 384d embeddings for all database questions...")
# Encode all questions into a NumPy Matrix (Shape: N x 384)
question_embeddings = model.encode(questions_list, show_progress_bar=True, convert_to_tensor=True)

print("Embeddings Matrix Shape:", question_embeddings.shape)

Total Unique Database Questions: 4966
Generating 384d embeddings for all database questions...


Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Embeddings Matrix Shape: torch.Size([4966, 384])


In [11]:
def search_similar_questions(user_query, top_k=5, threshold=0.55):
    """
    Given a user question:
    1. Embed query
    2. Compute cosine similarity against all stored embeddings
    3. Return Top K most similar questions with confidence scores & threshold decision
    """
    # Embed user question
    query_embedding = model.encode(user_query, convert_to_tensor=True)
    
    # Calculate similarity across all database questions
    cosine_scores = util.cos_sim(query_embedding, question_embeddings)[0]
    
    # Get top_k indices sorted descending
    top_results = torch.topk(cosine_scores, k=top_k)
    
    print(f"\nUSER QUERY: '{user_query}'")
    print("=" * 60)
    
    results = []
    for score, idx in zip(top_results[0], top_results[1]):
        matched_question = questions_list[idx]
        sim_percentage = score.item() * 100
        is_duplicate = score.item() >= threshold
        
        results.append({
            'question': matched_question,
            'similarity': sim_percentage,
            'status': 'DUPLICATE' if is_duplicate else 'NEW QUESTION'
        })
        
        status_flag = "DUPLICATE" if is_duplicate else "⚪ LOW MATCH"
        print(f"{status_flag} [{sim_percentage:.1f}% Match] → {matched_question}")
        
    return results

# Test the search engine with a user question!
test_query = "What is the best way to learn Python for beginners?"
search_similar_questions(test_query, top_k=5)


USER QUERY: 'What is the best way to learn Python for beginners?'
DUPLICATE [93.4% Match] → How should I begin learning Python?
DUPLICATE [92.3% Match] → How should I start learning Python?
DUPLICATE [87.5% Match] → How do I learn Python systematically?
DUPLICATE [63.8% Match] → Starting with no programming experience, how long will it take to learn Python 3?
DUPLICATE [63.3% Match] → How should you start learning programming?


[{'question': 'How should I begin learning Python?',
  'similarity': 93.43621730804443,
  'status': 'DUPLICATE'},
 {'question': 'How should I start learning Python?',
  'similarity': 92.339688539505,
  'status': 'DUPLICATE'},
 {'question': 'How do I learn Python systematically?',
  'similarity': 87.45359182357788,
  'status': 'DUPLICATE'},
 {'question': 'Starting with no programming experience, how long will it take to learn Python 3?',
  'similarity': 63.849860429763794,
  'status': 'DUPLICATE'},
 {'question': 'How should you start learning programming?',
  'similarity': 63.280189037323,
  'status': 'DUPLICATE'}]